In [ ]:
import os
import requests
import io
import pandas as pd
import itertools
from sklearn.metrics import brier_score_loss
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold


In [ ]:
# Load the processed tournament data

api_url = "https://api.github.com/repos/COMP3608-GROUP-9-PROJECT/COMP3608_PROJECT/contents/ProcessedData/processed_tournament_data.csv"
os.makedirs("processed", exist_ok=True)

response = requests.get(api_url)

if response.status_code == 200:
    file = response.json()
    download_url = file["download_url"]

    csv_data = requests.get(download_url).content
    tournament_data = pd.read_csv(io.BytesIO(csv_data))

    print("Loaded into DataFrame successfully")
else:
    raise Exception(f"Failed to access API: {response.status_code}")

# Define target
y = tournament_data["Win"]


# Define features
features = [col for col in tournament_data.columns if col not in ["Win", "Season", "PointDifference", "Division", "SeedDifference","RatingDifference"]] # Changed df.columns to tournament_data.columns and corrected 'Divison' to 'Division'


# Create dataframe to hold features
X = tournament_data[features]

groups = tournament_data["Season"] # Changed df["Season"] to tournament_data["Season"]
seasons = tournament_data["Season"].unique() # Changed df["Season"] to tournament_data["Season"]

gkf = GroupKFold(n_splits=len(seasons))

In [ ]:
# Initialize a list to store Brier scores for each fold
brier_scores = []

# Iterate over each fold in Group K-Fold cross-validation
for fold, (train_index, val_index) in enumerate(gkf.split(X, y, groups)):
    print(f"--- Fold {fold+1}/{len(seasons)} ---")

    # Split data into training and validation sets for the current fold
    train_X, val_X = X.iloc[train_index], X.iloc[val_index]
    train_y, val_y = y.iloc[train_index], y.iloc[val_index]

    # Create and train the Random Forest model for the current fold
    rf_model = RandomForestClassifier(n_estimators=100, random_state=1)
    rf_model.fit(train_X, train_y)

    # Predict probabilities for the validation set of the current fold
    rf_val_predictions = rf_model.predict_proba(val_X)[:, 1]

    # Evaluate predictions using Brier Score for the current fold
    fold_brier_score = brier_score_loss(val_y, rf_val_predictions)
    brier_scores.append(fold_brier_score)
    print(f"Random Forest Validation Brier Score for Fold {fold+1}: {fold_brier_score:.4f}")

# Calculate and print the average Brier score across all folds
mean_brier_score = sum(brier_scores) / len(brier_scores)
print(f"\nAverage Random Forest Validation Brier Score across all folds: {mean_brier_score:.4f}")

Random Forest Validation Brier Score: 0.1789


In [ ]:
api_url = "https://api.github.com/repos/COMP3608-GROUP-9-PROJECT/COMP3608_PROJECT/contents/ProcessedData/Sample_Submission.csv"
os.makedirs("processed", exist_ok=True)

response = requests.get(api_url)

if response.status_code == 200:
    file = response.json()
    download_url = file["download_url"]

    csv_data = requests.get(download_url).content
    sample_submission = pd.read_csv(io.BytesIO(csv_data))

    print("Loaded into DataFrame successfully")
else:
    raise Exception(f"Failed to access API: {response.status_code}")


In [ ]:
x_new = sample_submission[features]
sample_submission["Pred"] = rf_model.predict_proba(x_new)[:, 1]
sample_submission[['ID', 'Pred']].to_csv("RandomForestPredictions.csv", index=False)
print("Predictions for 2025 matchups saved to RandomForestPredictions.csv")

Predictions for 2025 matchups saved to RandomForestPredictions.csv
